# Package file diagnostics

Place this notebook in `Code_Release_Candidate` and run its code cell. It distinguishes missing files, changed content, and notebook metadata/output changes. Existing files and experiment outputs are not modified.


In [1]:
EXPECTED_MANIFEST_SHA256 = 'f0d109b064e3dc0270d39c04e46227a26248b01b6ed2e9aeb4618475cc3d900a'
EXPECTED_NOTEBOOK_CONTENT = {'00_Check_Recorded_Run.ipynb': '3544e37a32a8d4e7a0bd4ba616bb7dbcaa4310cfb6a376d1600659eb76808275', 'notebooks/07_fd001_pca_comparison.ipynb': '9eeb1366854dffa19818dd1fa0b1647e784194fe576e8cb5823459d0868dc971', 'notebooks/02_fd001_engine_protocol.ipynb': 'af66647ad782791c136623136db42f761f8db0343ae3e6ec28c19ffb30022093', 'notebooks/10C_additional_five_detector_cv.ipynb': '562f7030d3ec63b7c3dbb8a0be8c9bf2ceeca9e14be577b03d4eb7dd350bcefd', 'notebooks/03_fd001_initial_pilot.ipynb': '5a7176cd3b35c0ec00b6e014fdcc0eba8f58f7ace63e396b62886930de09bc4e', 'notebooks/02B_recorded_run_audit.ipynb': 'c7e2967c32144bbec3e855a0c8357674fe9bbaa913720cb4f8cd05085fb49e2b', 'notebooks/15A_sead_preflight.ipynb': '9d67410d29b491f35c43a64622f110d22b22903d5a5302a3c2f178d484bb3deb', 'notebooks/03B_fd001_five_detector_pilot.ipynb': 'fa94ec7f70d3c1cd428d4b4597af03bce07327fbe3e389b9e5bd24179f6c1c96', 'notebooks/10B_additional_pilot_historical.ipynb': '9a3e0557109ef8dcb29f31bf1ee63f02a002202dd639731ae13be8c883f204a4', 'notebooks/05B_fd001_five_detector_cv.ipynb': 'ed50fae54e0f559cffdf78a538c59bd2f4a48ece343e11190a95a3282d7772d8', 'notebooks/09_additional_engine_protocol.ipynb': '6a5bea5bfba4b1f90dfbb212f37950b0d264a6b611b815ead51fac80f3586db8', 'notebooks/10_additional_representations.ipynb': 'a6f879ceb429b18b8d83fbb7add5a53c578b025c695c3b68cf50f788cee439d5', 'notebooks/04_fd001_ae_pilot.ipynb': '945e2ae3461638535827124da61afef0e951b836a13123fc980011db017633c5', 'notebooks/11_detector_roc.ipynb': '9543cfd5fa4c9fd80aaeb5d3c53d11f6b9061a7d8ec9c856c58cd068ae5f428e', 'notebooks/07C_fd001_pca_five_detector_cv.ipynb': '5ef8713a0ca74b33c9e7e7a2e2a8970f4a11cbe4dae62e4f356315f9075c4c0c', 'notebooks/15C_sead_full_cv.ipynb': 'b7f8e5ab49334a90dc7656ba46182fbe74829e891cb95d9e578ce112f01ec7ed', 'notebooks/02C_recorded_score_check.ipynb': '64098a9eb042e700bc5c693d394f7f654c18fbbfbb1bc11de770f06e9726bae0', 'notebooks/13_representation_tsne.ipynb': 'f42a00796e913b231b49e046cf06c30e37a020fcd9c8617cede5ea22ae2d394b', 'notebooks/06C_fd001_ae_five_detector_cv.ipynb': '7134918820bd76ff1c9450ae98dd12b0a84e08d7dddb3045d2710b5ed388692a', 'notebooks/07B_fd001_pca_five_detector_pilot.ipynb': '2580398e15c121efc76c9ca51f18bb9c54b8680abf48b837c01242ea914d145a', 'notebooks/01_fd001_data_audit.ipynb': 'ee41d021d21cbcc3603246170b33f3f40b8c72ff29e68cf9872e5e3ff2e3989d', 'notebooks/10B1_additional_five_detector_pilot.ipynb': '183052d5602f134f7175aeeb6718b5d0656a12d51a3245170f76122214ec3a5d', 'notebooks/15B_sead_training_pilot.ipynb': '7f06b6499f59cd044f7c265b29768cdb85ff2cae7a08e31746335951de3414d6', 'notebooks/05_fd001_representation_cv.ipynb': 'de320bf87cf4c2f65bb8cd3e1bb1379bc60d0c2eec5e85913913f8e441dc163a', 'notebooks/06B_fd001_ae_five_detector_pilot.ipynb': 'ac5054d555b6e96ef7ce92240153c42d6395290268bfcc332d69c936426449ea', 'notebooks/06_fd001_ae_ablation.ipynb': 'f2c5f5e63f2bd5cdd49e0f975181d3989337ab55b4cef9b826fc457a1dcfbd97', 'notebooks/08_additional_data_audit.ipynb': '74e803c3cac1e87c6a2e096cbf6ad0527175823bb373190f07dcded71b46dc03', 'notebooks/14_detector_shap.ipynb': 'c64e77c0e2a98e38ed67b8c02fb37984cb73bd6d804fcf5a6fcc9b15babf6ec6', 'notebooks/12_fixed_selected_parameters.ipynb': 'd71a0df26fd0eb779d5e935b0d4f1bd4e0aaffaf629b5f07c2738135533450cd'}
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def notebook_content_sha256(path):
    notebook = json.loads(path.read_text(encoding='utf-8'))
    cells = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))}
             for c in notebook['cells']]
    text = json.dumps(cells, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(text.encode()).hexdigest()


def diagnose(candidate_dir=None):
    if candidate_dir is None:
        candidates = [Path.cwd(), *Path.cwd().parents]
        candidate_dir = next((p for p in candidates if (p/'package_manifest.json').is_file()
                              and (p/'release_runtime.py').is_file()), None)
    if candidate_dir is None:
        raise FileNotFoundError('Place this notebook in Code_Release_Candidate, then run it.')
    root = Path(candidate_dir).expanduser().absolute()
    manifest_path = root/'package_manifest.json'
    manifest_hash = sha256(manifest_path)
    report = {'stage': '16B_package_file_diagnostics', 'candidate_directory': str(root),
              'manifest_matches_delivered_package': manifest_hash == EXPECTED_MANIFEST_SHA256,
              'manifest_sha256': manifest_hash,
              'model_training_performed': False, 'existing_files_modified': False,
              'experiment_source_hashes_rechecked': False}
    if not report['manifest_matches_delivered_package']:
        report.update(status='MANIFEST_DIFFERS', issues=[])
    else:
        records = []
        for item in json.loads(manifest_path.read_text())['files']:
            p = root/item['path']
            row = {'path': item['path'], 'expected_sha256': item['sha256']}
            if not p.resolve().is_relative_to(root.resolve()):
                row['status'] = 'OUTSIDE_CANDIDATE_FOLDER'
            elif not p.exists():
                row['status'] = 'MISSING'
            elif not p.is_file():
                row['status'] = 'NOT_A_FILE'
            else:
                try:
                    row.update(actual_sha256=sha256(p), size_bytes=p.stat().st_size)
                    row['status'] = 'MATCH' if row['actual_sha256']==item['sha256'] else 'CONTENT_CHANGED'
                    if row['status']=='CONTENT_CHANGED' and item['path'] in EXPECTED_NOTEBOOK_CONTENT:
                        row['notebook_content_sha256'] = notebook_content_sha256(p)
                        if row['notebook_content_sha256']==EXPECTED_NOTEBOOK_CONTENT[item['path']]:
                            row['status'] = 'NOTEBOOK_METADATA_OR_OUTPUT_ONLY'
                except (OSError, ValueError, KeyError) as exc:
                    row.update(status='READ_OR_PARSE_ERROR', detail=str(exc))
            records.append(row)
        report.update(status='PACKAGE_MATCH' if all(r['status']=='MATCH' for r in records) else 'PACKAGE_DETAILS_AVAILABLE',
                      counts=dict(Counter(r['status'] for r in records)),
                      issues=[r for r in records if r['status']!='MATCH'])
        report['folder_layout'] = {}
        for name in ['', 'notebooks', 'supplementary']:
            folder = root/name
            report['folder_layout'][name or '.'] = {'exists': folder.is_dir(),
                'entries': sorted(p.name for p in folder.iterdir())[:150] if folder.is_dir() else []}
    output = root/'package_diagnostics'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    output.mkdir(parents=True, exist_ok=False)
    summary = output/'package_file_diagnostics.json'
    summary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print('STATUS:', report['status'])
    print('COUNTS:', report.get('counts', {}))
    print('SEND THIS FILE:', summary)
    return report


report = diagnose()


STATUS: PACKAGE_DETAILS_AVAILABLE
COUNTS: {'MISSING': 1, 'MATCH': 93}
SEND THIS FILE: /Users/emredelibas/Documents/Akademik/Python/ServetMakale/Code_Release_Candidate/package_diagnostics/20261106T180752_871937Z/package_file_diagnostics.json


In [2]:
from pathlib import Path
import hashlib

folder = Path(
    "/Users/emredelibas/Documents/Akademik/Python/"
    "ServetMakale/Code_Release_Candidate"
)
assert (folder / "package_manifest.json").is_file(), "Paket klasörü bulunamadı."

content = (
    "settings.json\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n"
    "data/\nrevision_outputs/\n.venv/\nvenv/\n"
).encode()

path = folder / ".gitignore"
if not path.exists():
    path.write_bytes(content)

assert hashlib.sha256(path.read_bytes()).hexdigest() == (
    "769e1b96c0deb381959a0d51296ac14448d738a413434b13e854e6a02d39e7d9"
), "Dosya içeriği beklenenle eşleşmiyor."

print("Eksik paket dosyası tamamlandı ve doğrulandı.")

Eksik paket dosyası tamamlandı ve doğrulandı.
